# Unity Catalog

Unity Catalog (UC) is Databricks's unified governance layer for all data and AI assets. It provides a single metastore that spans multiple workspaces, centralises access control, and adds data lineage, auditing, and discovery on top of the standard Delta Lake storage model.

This notebook covers:
- The three-level namespace: catalog → schema → table/view/function
- Metastore, catalog, and schema hierarchy
- Securable objects and privilege model
- GRANT, REVOKE, and SHOW GRANTS
- Data lineage and audit logging
- Unity Catalog vs legacy Hive metastore
- Cluster access modes and Unity Catalog compatibility
- External locations and storage credentials
- System tables

## 1. The Three-Level Namespace

Unity Catalog organises all data assets in a three-level hierarchy:

```
metastore
└── catalog
    └── schema  (also called database)
        ├── table
        ├── view
        ├── function
        └── volume
```

A fully qualified object name is: `catalog.schema.object_name`

```sql
SELECT * FROM main.bronze.orders;
SELECT * FROM analytics.gold.customer_lifetime_value;
```

You can set a default catalog and schema to avoid typing the full path every time:

```sql
USE CATALOG main;
USE SCHEMA bronze;
-- now you can write:
SELECT * FROM orders;
```

**Exam tip:** The three-level namespace is a Unity Catalog concept. The legacy Hive metastore uses a two-level namespace: `schema.table` (with `hive_metastore` as an implicit catalog).

## 2. Metastore, Catalog, Schema

### Metastore
- One metastore per Databricks account region (typically)
- Stores all metadata: table schemas, access policies, lineage, audit logs
- Managed by account admins
- Can be attached to multiple workspaces — this is how cross-workspace data sharing works

### Catalog
- Top-level namespace container
- Created and owned by metastore admins or users with `CREATE CATALOG` privilege
- Two special catalogs exist in every metastore:
  - **`hive_metastore`** — the legacy metastore, always present for backwards compatibility
  - **`system`** — built-in catalog containing operational and audit tables

### Schema
- Groups tables, views, functions, and volumes within a catalog
- Has an associated storage location (optional — defaults to the catalog's storage location)
- Managed tables in the schema are stored under the schema's storage path

### Volume
- A UC object representing a cloud storage path
- Accessed as `/Volumes/catalog/schema/volume_name/path/to/file`
- Governed by UC access control — replaces direct `dbfs:/` paths in UC-enabled workspaces

## 3. Securable Objects and the Privilege Model

Every object in Unity Catalog is a **securable** — you can GRANT and REVOKE privileges on it.

### Securable Object Hierarchy

```
metastore
  catalog
    schema
      table / view / function / volume
```

Privileges are inherited downward. Granting `USE CATALOG` on a catalog does not automatically grant access to tables — you must also grant `USE SCHEMA` on the schema and then a data privilege (SELECT, MODIFY, etc.) on the table.

### Key Privileges

| Privilege | Applies to | What it allows |
|-----------|------------|---------------|
| `USE CATALOG` | catalog | Enter the catalog namespace |
| `CREATE CATALOG` | metastore | Create new catalogs |
| `USE SCHEMA` | schema | Enter the schema namespace |
| `CREATE SCHEMA` | catalog | Create schemas in the catalog |
| `CREATE TABLE` | schema | Create tables in the schema |
| `SELECT` | table/view | Read data |
| `MODIFY` | table | INSERT, UPDATE, DELETE, MERGE |
| `ALL PRIVILEGES` | any | All privileges on the object |
| `READ VOLUME` | volume | Read files from the volume |
| `WRITE VOLUME` | volume | Write files to the volume |

**Exam tip:** To allow a user to query a table, you need at minimum three grants:
1. `USE CATALOG` on the catalog
2. `USE SCHEMA` on the schema
3. `SELECT` on the table

## 4. GRANT, REVOKE, SHOW GRANTS

In [ ]:
# ── Setup demo schema ─────────────────────────────────────────────────────────
spark.sql("CREATE CATALOG IF NOT EXISTS demo_uc")
spark.sql("CREATE SCHEMA IF NOT EXISTS demo_uc.sales")
spark.sql("""
    CREATE TABLE IF NOT EXISTS demo_uc.sales.orders (
        order_id   BIGINT,
        customer   STRING,
        amount     DOUBLE
    )
""")

In [ ]:
# ── GRANT: minimum set to allow a user to SELECT from a table ─────────────────
spark.sql("GRANT USE CATALOG ON CATALOG demo_uc TO `analysts`")
spark.sql("GRANT USE SCHEMA  ON SCHEMA  demo_uc.sales TO `analysts`")
spark.sql("GRANT SELECT      ON TABLE   demo_uc.sales.orders TO `analysts`")

# Grant MODIFY so they can INSERT/UPDATE/DELETE too
spark.sql("GRANT MODIFY ON TABLE demo_uc.sales.orders TO `analysts`")

In [ ]:
# ── SHOW GRANTS: inspect who has access ───────────────────────────────────────
spark.sql("SHOW GRANTS ON TABLE demo_uc.sales.orders").show(truncate=False)
spark.sql("SHOW GRANTS ON SCHEMA demo_uc.sales").show(truncate=False)
spark.sql("SHOW GRANTS ON CATALOG demo_uc").show(truncate=False)

In [ ]:
# ── REVOKE ────────────────────────────────────────────────────────────────────
spark.sql("REVOKE MODIFY ON TABLE demo_uc.sales.orders FROM `analysts`")

# Verify SELECT still works, MODIFY is gone
spark.sql("SHOW GRANTS ON TABLE demo_uc.sales.orders").show(truncate=False)

In [ ]:
# ── Grant at schema level — propagates to all tables in schema ────────────────
spark.sql("GRANT SELECT ON SCHEMA demo_uc.sales TO `data_consumers`")
# Note: still need USE CATALOG + USE SCHEMA for the principal to navigate here

## 5. Principals: Users, Groups, and Service Principals

You grant privileges to **principals**:

| Principal type | Syntax in SQL | Notes |
|----------------|---------------|-------|
| User | `` `user@company.com` `` | Individual account user |
| Group | `` `analysts` `` | Managed in account console or synced from IdP |
| Service principal | `` `sp-abc123` `` | Non-human identity for automation |

**Best practice:** Grant privileges to **groups**, not individual users. Add users to groups. This makes access management scalable — revoking a user's access is a group membership change, not a hunt through every table's grants.

### Account-Level Groups vs Workspace-Level Groups
- **Account groups** are managed at the account level and visible across all workspaces that share the metastore
- **Workspace-local groups** exist only within one workspace — they cannot be granted Unity Catalog privileges

**Exam tip:** To use a group as a UC principal, it must be an **account-level group**, not a workspace-local group.

## 6. Table Ownership and the Owner Privilege

Every securable has an **owner**. The owner can:
- Grant and revoke privileges on that object
- Transfer ownership to another principal
- DROP the object

Ownership is set at creation time to the creating principal. You can transfer it:

```sql
ALTER TABLE demo_uc.sales.orders OWNER TO `data-engineering-team`;
ALTER SCHEMA demo_uc.sales OWNER TO `data-engineering-team`;
ALTER CATALOG demo_uc OWNER TO `platform-admin-group`;
```

**Exam tip:** Only the **object owner** or a **metastore admin** can transfer ownership.

## 7. Data Lineage

Unity Catalog automatically captures column-level data lineage — which tables and columns a given column derives from, and which downstream tables consume it.

Lineage is captured for:
- SQL queries that create tables (`CREATE TABLE AS SELECT`, `INSERT INTO ... SELECT`)
- DataFrame writes via `.saveAsTable()` and `.toTable()`
- Delta Live Tables pipelines

Lineage is visible in the **Catalog Explorer** UI: navigate to a table → Lineage tab. You can trace upstream sources and downstream consumers without writing any queries.

Lineage data is also available programmatically via the [REST API](https://docs.databricks.com/api/workspace/lineagetracking) and in the system tables.

**Exam tip:** Lineage is **automatic** — no configuration required beyond using Unity Catalog. It does not require any additional privileges to view lineage for tables you already have SELECT access to.

## 8. Audit Logging

Unity Catalog logs every data access event — SELECT, INSERT, MERGE, GRANT, REVOKE, schema changes — to audit logs.

Audit logs are available in two ways:

### System Tables
```sql
-- Recent table reads in the last 24 hours
SELECT
  event_time,
  user_identity.email       AS user,
  request_params.table_name AS table_name,
  action_name
FROM system.access.audit
WHERE event_time >= NOW() - INTERVAL 1 DAY
  AND action_name IN ('commandSubmit', 'selectTable')
ORDER BY event_time DESC
LIMIT 50
```

### Audit Log Delivery to Cloud Storage
You can configure Databricks to deliver audit logs to your own S3/ADLS bucket in JSON format for long-term retention and SIEM integration.

**Exam tip:** `system.access.audit` is the table to query for access events. `system.lakeflow.job_runs` is for job run history. Both require the `system` catalog to be enabled for your metastore.

## 9. External Locations and Storage Credentials

### Storage Credential
A UC object that wraps a cloud IAM role or managed identity giving Databricks permission to access a cloud storage bucket. Created by account admins.

```sql
-- View storage credentials you have access to
SHOW STORAGE CREDENTIALS;
```

### External Location
A UC object that maps a cloud storage path (`s3://my-bucket/data/`) to a named, governable location using a storage credential.

```sql
-- View external locations
SHOW EXTERNAL LOCATIONS;

-- Check what path an external location covers
DESCRIBE EXTERNAL LOCATION my_external_location;
```

### External Tables
An external table in Unity Catalog points to data at an external location. The data is **not** managed by UC — dropping the table does not delete the files.

```sql
CREATE TABLE demo_uc.sales.external_orders
LOCATION 's3://my-bucket/orders/'
AS SELECT * FROM demo_uc.sales.orders;
```

**Exam tip:** In UC, you cannot read or write to a cloud path unless either:
- The path is under a registered External Location, **or**
- The path is exposed as a Volume

Direct `s3://` paths in notebooks are blocked by UC's network policy when the cluster uses single-user or shared access mode.

## 10. Cluster Access Modes and UC Compatibility

| Access Mode | UC Support | Notes |
|-------------|------------|-------|
| **Single User** | Full | One user per cluster; all UC features available |
| **Shared** | Full | Multiple users; UC enforces per-user access control |
| **No Isolation Shared** | None | No UC support; legacy mode; not recommended |

**Exam tip:** The `No Isolation Shared` access mode does **not** support Unity Catalog. For UC, always use Single User or Shared access mode.

DLT pipelines also require Unity Catalog-compatible access — configure the pipeline's target catalog and schema in the pipeline settings.

### Runtime Requirement
Unity Catalog requires Databricks Runtime 11.3 LTS or higher. Clusters on older runtimes cannot connect to UC-governed resources.

## 11. Unity Catalog vs Legacy Hive Metastore

| Feature | Unity Catalog | Legacy Hive Metastore |
|---------|---------------|-----------------------|
| Namespace | 3-level: catalog.schema.table | 2-level: schema.table |
| Scope | Account-wide (all workspaces) | Single workspace |
| Access control | Fine-grained SQL GRANT/REVOKE | Table ACLs (limited, deprecated) |
| Column-level security | Yes | No |
| Row-level security | Yes (via row filters) | No |
| Data lineage | Automatic | No |
| Audit logs | System tables + delivery | Limited |
| Delta Sharing | Native | Not supported |
| Volumes | Yes | No |

The legacy Hive metastore is accessible as the `hive_metastore` catalog in UC-enabled workspaces. Existing tables in `hive_metastore` are not automatically governed by UC — you must upgrade them by recreating them in a UC catalog.

**Exam tip:** `hive_metastore` always exists as a special catalog. You can query it, but UC governance (lineage, fine-grained access control) does not apply to objects inside it.

## 12. Dynamic Views — Row and Column Filtering

Unity Catalog supports row-level and column-level security through dynamic views. A dynamic view uses `current_user()` or `is_account_group_member()` to filter rows or mask columns based on the identity of the querying user.

### Column Masking Example

In [ ]:
# Create a view that masks the customer email for non-admins
spark.sql("""
    CREATE OR REPLACE VIEW demo_uc.sales.orders_masked AS
    SELECT
        order_id,
        CASE
            WHEN is_account_group_member('data-admins') THEN customer
            ELSE CONCAT(LEFT(customer, 2), '****')
        END AS customer,
        amount
    FROM demo_uc.sales.orders
""")

# Grant SELECT on the view to analysts (NOT the underlying table)
spark.sql("GRANT SELECT ON VIEW demo_uc.sales.orders_masked TO `analysts`")

In [ ]:
# Row-level security: analysts can only see their region's orders
# Assume orders table has a `region` column
spark.sql("""
    CREATE OR REPLACE VIEW demo_uc.sales.orders_regional AS
    SELECT * FROM demo_uc.sales.orders
    WHERE
        is_account_group_member('data-admins')  -- admins see all rows
        OR region = current_user()              -- others see only their region
""")

**Exam tip:** Dynamic views are evaluated at query time with the identity of the querying user, not the view creator. This makes them the standard pattern for fine-grained data access without duplicating tables.

## 13. Cleanup

In [ ]:
spark.sql("DROP CATALOG IF EXISTS demo_uc CASCADE")

## 14. Key Exam Topics Summary

| Topic | What to remember |
|-------|------------------|
| 3-level namespace | `catalog.schema.table` — UC only |
| Minimum grants to query a table | `USE CATALOG` + `USE SCHEMA` + `SELECT` |
| Account-level groups | Required for UC grants — workspace-local groups cannot be used |
| `hive_metastore` | Always exists; not UC-governed; 2-level namespace |
| No Isolation Shared | Does NOT support UC |
| Lineage | Automatic, no config needed |
| Audit logs | `system.access.audit` table |
| External Location | Required to access cloud paths in UC |
| Dynamic views | Row/column security via `current_user()` / `is_account_group_member()` |
| Ownership | Owner or metastore admin can transfer ownership |